### ECG code

Batch ECG plot generator (one plot per patient file)

This script:
1) Walks a ROOT folder containing one subfolder per patient.
2) Finds ECG text files inside each patient folder.
3) Parses lines of the form: "HH:MM:SS,mmm; value"
4) Converts time-of-day timestamps to "seconds since first sample".
5) Optionally crops to first N seconds and/or downsamples for speed.
6) Plots signal vs time and saves an image for each patient/file.

Requirements:
pip install numpy matplotlib

In [7]:
import re
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt

# =========================
# CONFIG
# =========================
ROOT_DIR = Path(r"C:\Users\JLOR0029\Desktop\Patients_ECG")
OUTPUT_DIR = Path(r"C:\Users\JLOR0029\Desktop\Patients_ECG\ecg_plots")     # writable folder
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

DPI = 150
MAX_SECONDS_TO_PLOT = None  # e.g. 10 or None

# Match: "HH:MM:SS,mmm; value"
LINE_RE = re.compile(
    r"^\s*(\d{2}):(\d{2}):(\d{2}),(\d{3})\s*;\s*([-+]?\d+(?:\.\d+)?)\s*$"
)

# =========================
# FILE HELPERS
# =========================
def iter_patient_folders(root):
    for p in root.iterdir():
        if p.is_dir() and not p.name.startswith("_"):
            yield p

def find_ecg_txt(patient_dir):
    for f in patient_dir.iterdir():
        if f.is_file() and f.name.lower() == "ecg.txt":
            return f
    return None

def safe_name(s):
    return re.sub(r"[^A-Za-z0-9_.-]+", "_", s)

# =========================
# PARSING
# =========================
def parse_ecg_elapsed_time(file_path):
    """
    Returns:
      t_sec : elapsed time in seconds starting at 0 (float)
      y     : signal values
    """
    t_ms = []
    y = []

    with file_path.open("r", encoding="utf-8", errors="ignore") as f:
        for line in f:
            m = LINE_RE.match(line)
            if not m:
                continue

            hh = int(m.group(1))
            mm = int(m.group(2))
            ss = int(m.group(3))
            ms = int(m.group(4))
            val = float(m.group(5))

            # milliseconds since midnight
            ms_midnight = ((hh * 60 + mm) * 60 + ss) * 1000 + ms
            t_ms.append(ms_midnight)
            y.append(val)

    if not t_ms:
        raise ValueError("No valid ECG samples found")

    # unwrap midnight rollover
    DAY_MS = 24 * 3600 * 1000
    unwrapped = [t_ms[0]]
    offset = 0

    for prev, curr in zip(t_ms[:-1], t_ms[1:]):
        if curr < prev:
            offset += DAY_MS
        unwrapped.append(curr + offset)

    t_ms = np.array(unwrapped, dtype=float)
    y = np.array(y, dtype=float)

    # elapsed time in seconds starting at 0
    t_sec = (t_ms - t_ms[0]) / 1000.0

    return t_sec, y

# =========================
# PLOTTING
# =========================
def estimate_fs(t):
    if len(t) < 2:
        return None
    dt = np.diff(t)
    dt = dt[dt > 0]
    if len(dt) == 0:
        return None
    return 1.0 / np.median(dt)

def plot_ecg(t, y, out_path, title):
    fs = estimate_fs(t)

    plt.figure(figsize=(12, 4))
    plt.plot(t, y, linewidth=0.8)

    plt.xlabel("Time (s)")
    plt.ylabel("Amplitude (raw units)")
    if fs:
        plt.title(f"{title} | ~{fs:.1f} Hz")
    else:
        plt.title(title)

    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig(out_path, dpi=DPI)
    plt.close()

# =========================
# MAIN
# =========================
def main():
    saved = 0
    errors = []

    for patient_dir in iter_patient_folders(ROOT_DIR):
        patient_id = patient_dir.name
        ecg_file = find_ecg_txt(patient_dir)

        if ecg_file is None:
            errors.append((patient_id, "ECG.txt not found"))
            continue

        try:
            t, y = parse_ecg_elapsed_time(ecg_file)

            if MAX_SECONDS_TO_PLOT is not None:
                mask = t <= MAX_SECONDS_TO_PLOT
                t = t[mask]
                y = y[mask]

            out_path = OUTPUT_DIR / safe_name(f"{patient_id}_ECG.png")
            plot_ecg(t, y, out_path, f"{patient_id} - ECG")

            saved += 1

        except Exception as e:
            errors.append((patient_id, str(e)))

    print(f"Saved {saved} ECG plots to {OUTPUT_DIR}")
    if errors:
        print("\nIssues:")
        for pid, msg in errors:
            print(f" - {pid}: {msg}")

if __name__ == "__main__":
    main()


Saved 1 ECG plots to C:\Users\JLOR0029\Desktop\Patients_ECG\ecg_plots

Issues:
 - ecg_plots: ECG.txt not found
